## Baseline scores on semart using CLIP

In [1]:
%load_ext autoreload
%autoreload 2

import torch.nn.functional as F

import open_clip 
import numpy as np
import pandas as pd

from src.model import SheafMultimodalGNN
from src.utils import *
from src.data import *
from torch_geometric.data import DataLoader
from src.metrics import *

#triplets = '../artistic_sheaf/data/testing_elements.json'
triplets = 'data/hertziana/test_set.json'
loaded_data = load_json_data(triplets)#[:5000]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

/home/ludosc/ludosc/conda/conda/envs/test/lib/python3.11/site-packages/lightning_fabric/__init__.py:29: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __import__("pkg_resources").declare_namespace(__name__)
/home/ludosc/ludosc/conda/conda/envs/test/lib/python3.11/site-packages/fairscale/experimental/nn/offload.py:19: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_fwd(orig_func)  # type: ignore
/home/ludosc/ludosc/conda/conda/envs/test/lib/python3.11/site-packages/fairscale/experimental/nn/offload.py:30: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  return torch.cuda.amp.custom_b

Loaded 3007 triplets from data/hertziana/test_set.json


In [2]:
device = 'cuda' if torch.cuda.is_available() else 'mps'
print(f"Using device: {device}")
seed_everything(seed=42)

# Load tokenizer and preprocessing
tokenizer = open_clip.get_tokenizer('ViT-B-32')
_, _, preprocess = open_clip.create_model_and_transforms('ViT-B-32', pretrained='laion2b_s34b_b79k')

# Initialize the model
model = SheafMultimodalGNN(
    latent_dim=512,
    edge_attr_dim=512,
    num_layers=3,
    step_size=1.0,
    lr=1e-4,
    test=False,
    #clip_grad=False,
    device='cuda' if torch.cuda.is_available() else 'mps'
)
    
# Load checkpoint
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=10-val_loss=9.17.ckpt", map_location=device) # SheafCLIP (trained CLIP 1/5, new eval)

checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=23-val_loss=5.44.ckpt", map_location=device) # new version no training
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=16-val_loss=9.26.ckpt", map_location=device) # new version
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=05-val_loss=9.31.ckpt", map_location=device) # new version
# checkpoint = torch.load("checkpoints/sheaf-gnn-epoch=03-val_loss=9.26.ckpt", map_location=device)
	
model.load_state_dict(checkpoint['state_dict'])
model = model.to(device)
model.eval()
print()

Using device: cuda



### Testing on normal dataset

In [3]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data, preprocess, tokenizer, base_folder='../', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

100%|██████████| 3007/3007 [00:03<00:00, 769.05it/s]


3007


/tmp/ipykernel_907882/3845338747.py:6: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)


In [5]:
#clip_texts = get_clip_texts(loaded_data, 'item2', get_tokenizer('ViT-B-32'), model)
clip_images = []
clip_texts = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, split='sheaf', check_images_=False)
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings = model(x_img, x_text, edge_index, edge_attr)
        print(embeddings.shape)
        clip_images.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
        clip_texts.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
        
clip_images = torch.cat(clip_images, dim=0)
clip_texts = torch.cat(clip_texts, dim=0)

print(f"Extracted {len(clip_texts)} text embeddings, each of shape {clip_texts[0].shape}")
print(f"Extracted {len(clip_images)} image embeddings, each of shape {clip_images[0].shape}")

clip_images = clip_images.cpu().detach().numpy()
clip_texts = clip_texts.cpu().detach().numpy()

torch.Size([795, 3, 224, 224]) torch.Size([2832, 77]) torch.Size([2, 3007]) torch.Size([3007, 77])
Checking maps tensor([[-0.0285],
        [-0.0297],
        [-0.0293],
        [-0.0267],
        [-0.0302]], device='cuda:0')
Checking maps tensor([[-0.0297],
        [-0.0173],
        [-0.0243],
        [-0.0241],
        [-0.0363]], device='cuda:0')
Checking maps tensor([[-0.6873],
        [ 0.0450],
        [-0.2633],
        [ 0.4422],
        [-0.7762]], device='cuda:0')
out shape torch.Size([3007, 512]) torch.Size([6014, 1])
Embeddings before change: tensor([[-0.1532, -0.2880,  0.4174, -0.1741,  0.7562],
        [-0.1532, -0.2880,  0.4174, -0.1741,  0.7562],
        [-0.1532, -0.2880,  0.4174, -0.1741,  0.7562],
        [-0.1532, -0.2880,  0.4174, -0.1741,  0.7562],
        [-0.0623, -0.1247,  0.4598,  0.1584,  0.0444]], device='cuda:0')
torch.Size([6014, 512])
Extracted 3007 text embeddings, each of shape torch.Size([512])
Extracted 3007 image embeddings, each of shape torch.Size

In [6]:
clip_images[:, :10]

array([[-0.05766555,  0.00367778,  0.04452692, ..., -0.0433471 ,
        -0.00365067,  0.06616417],
       [-0.05243047, -0.05825962,  0.00947672, ...,  0.00996302,
        -0.00778577,  0.00566261],
       [-0.05984964, -0.00126363,  0.05554506, ..., -0.05014165,
        -0.00489303,  0.06874438],
       ...,
       [-0.02518648, -0.00984123,  0.112614  , ..., -0.00468725,
        -0.01297714,  0.07085299],
       [-0.05148614,  0.02043425, -0.00589816, ..., -0.03644775,
        -0.01680818,  0.04666658],
       [-0.00551256, -0.01128687,  0.09076423, ...,  0.02286411,
        -0.03200402,  0.06008235]], dtype=float32)

In [7]:
clip_texts[:, :10]

array([[ 0.05745905,  0.0351387 , -0.1370369 , ...,  0.02326353,
        -0.02581201, -0.07656758],
       [-0.06198857, -0.06415029,  0.01322827, ...,  0.00139708,
        -0.00830864,  0.01277137],
       [-0.06124868,  0.00564822,  0.06060275, ..., -0.04672541,
         0.00285822,  0.06690233],
       ...,
       [ 0.00519443, -0.02270546,  0.08122143, ...,  0.02036674,
        -0.0356914 ,  0.0576378 ],
       [ 0.06205192,  0.03685603, -0.1404803 , ...,  0.02451018,
        -0.02501566, -0.07700841],
       [ 0.00045878, -0.02576286,  0.08210369, ...,  0.01549006,
        -0.04160285,  0.05879255]], dtype=float32)

In [8]:
# take a subset of the image embeddings and plot them with plotly interactively (in 2D using umap) showing the edge_index[0, i] on hover
#import umap
#import plotly.express as px
#reducer = umap.UMAP()
#from sklearn.decomposition import PCA
#reducer = PCA(n_components=2)

#embedding_2d = reducer.fit_transform(np.concatenate([clip_images, clip_texts], axis=0) ) # take only first 
#fig = px.scatter(x=embedding_2d[:, 0], y=embedding_2d[:, 1],
 #               hover_data=[np.concatenate([np.arange(len(clip_images)), np.arange(len(clip_texts))], axis=0),
  #                          np.concatenate([test_graph_data.edge_index[0, :len(clip_images)].cpu().numpy(), test_graph_data.edge_index[1, :len(clip_texts)].cpu().numpy()], axis=0)], 
  #              color=['images']*(len(clip_images)) + ['text']*(len(clip_texts)))
#fig.show()

### Image-to-text retrieval	and Text-to-image retrieval		
r@1	r@5	r@10	

In [9]:
adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data) 
print(f"Adjacency matrix shape: {adj_matrix.shape}")

Adjacency matrix shape: (2979, 2834)


In [10]:
sim_matrix = get_sim_matrix([t["item1"] + t["link"] for t in loaded_data], 
                            [t["item2"] + t["link"] for t in loaded_data], 
                            clip_images, clip_texts,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")

Similarity matrix shape: (2979, 2834)


In [14]:
results = compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])
recalls = [k for k in results.keys() if 'recall' in k and 'mean' not in k]
for rec in recalls:
    print(rec, results[rec])

t2i_recall@1 tensor(0.0042)
t2i_recall@5 tensor(0.0172)
t2i_recall@10 tensor(0.0356)
i2t_recall@1 tensor(0.0050)
i2t_recall@5 tensor(0.0180)
i2t_recall@10 tensor(0.0327)


In [12]:
#print('uniformity images', uniformity(torch.tensor(clip_images)))
#print('uniformity texts', uniformity(torch.tensor(clip_texts)))
#print('alignment', alignment(torch.tensor(clip_images), torch.tensor(clip_texts)))

In [15]:
recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=min(5, len(loaded_data)))

query_field = 'item1'  # image path
rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}
idx_to_img = {idx: img for img, idx in img_to_idx.items()}

for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
    query = idx_to_img[i]
    recommendations = [idx_to_txt[j] for j in rec_indices]
    print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
    print(f"Ground Truth: {[loaded_it[rec_field] for loaded_it in loaded_data if loaded_it['item1'] + loaded_it['link'] == query]}")
    print("Recommendations:")
    for rec in recommendations:
        print(f"  - {rec}")
    print("-" * 40)

Query: /Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/gemalde/bh000444paragraph foto EN
Ground Truth: ['A rare photograph of Alinari Anderson from 1886 is preserved in this albumen print. The photograph comes from an album created before 1957. The print is fully adhered, with the lower right corner slightly bent. There are also traces of fingerprints and adhesive residue. The image shows slight fading. On the back, the letter "Vs." is written in ballpoint pen. There is no stamp and the photograph is released.']
Recommendations:
  - The curator of the Florenz Gabinetto Disegni e Stampe degli Uffizi is Mr. Florenz. His object number is U 1222A.paragraph verwalter EN
  - The administrator is Florenz Gabinetto Disegni e Stampe degli Uffizi. The object is marked U 1874A.paragraph verwalter EN
  - The administrator of the German National Museum in Nuremberg is the Hz 4320/712a.paragraph verwalter EN
  - The administrator of the objects is the Kunstbibliothek of the Prussian Cultural Heritag

### Retrieval per type of relationship

In [16]:
from src.metrics import *

verbose = False
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    #if typ != 'timeframe':
    #    continue
    
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    indices_new = [i for i, l in enumerate(loaded_data) if l['link'] == typ]
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new) 
    #print(adj_matrix.shape, 'shape adj matrix', len(set([l['item2'] for l in loaded_data_new])), 'unique text values')
    
    # subset of clip_img and clip_texts for loaded_data_new with typ = typ withou img_to_idx and txt_to_idx
    clip_imgs_n = clip_images[indices_new]
    clip_txts_n = clip_texts[indices_new]
    
    #print(clip_imgs_n[:5, :10])
    #print(list(img_to_idx.items())[:5])
    #sim_matrix = clip_imgs_n @ clip_txts_n.T
    sim_matrix = get_sim_matrix([t["item1"] + t["link"] for t in loaded_data_new], 
                                [t["item2"] + t["link"] for t in loaded_data_new], 
                                clip_imgs_n, clip_txts_n,
                                img_to_idx, txt_to_idx)
    
    #print(f"Similarity matrix shape: {sim_matrix.shape}")
    #print(compute_clip_metrics(torch.tensor(clip_imgs_n), torch.tensor(clip_txts_n), topk=[1, 5, 10]))
    results = compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])
    recalls = [k for k in results.keys() if 'recall' in k and 'mean' not in k]
    for rec in recalls:
        print(rec, results[rec])
    #print('uniformity images', uniformity(torch.tensor(clip_imgs_n)))
    #print('uniformity texts', uniformity(torch.tensor(clip_txts_n)))
    #print('alignment', alignment(torch.tensor(clip_imgs_n), torch.tensor(clip_txts_n)))
    
    if verbose:
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=min(5, len(loaded_data_new)))

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}
        idx_to_img = {idx: img for img, idx in img_to_idx.items()}

        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = idx_to_img[i]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {[loaded_it[rec_field] for loaded_it in loaded_data_new if loaded_it['item1'] + loaded_it['link'] == query]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

Processing type: paragraph verwalter EN
t2i_recall@1 tensor(0.0027)
t2i_recall@5 tensor(0.0114)
t2i_recall@10 tensor(0.0252)
i2t_recall@1 tensor(0.0051)
i2t_recall@5 tensor(0.0203)
i2t_recall@10 tensor(0.0419)
Processing type: paragraph foto EN
t2i_recall@1 tensor(0.0012)
t2i_recall@5 tensor(0.0062)
t2i_recall@10 tensor(0.0125)
i2t_recall@1 tensor(0.0013)
i2t_recall@5 tensor(0.0063)
i2t_recall@10 tensor(0.0126)
Processing type: paragraph iconclass EN
t2i_recall@1 tensor(0.0050)
t2i_recall@5 tensor(0.0248)
t2i_recall@10 tensor(0.0604)
i2t_recall@1 tensor(0.0033)
i2t_recall@5 tensor(0.0166)
i2t_recall@10 tensor(0.0447)
Processing type: paragraph obj EN
t2i_recall@1 tensor(0.0088)
t2i_recall@5 tensor(0.0350)
t2i_recall@10 tensor(0.0701)
i2t_recall@1 tensor(0.0113)
i2t_recall@5 tensor(0.0397)
i2t_recall@10 tensor(0.0687)


## Evaluation with pseudo edge index

In [ ]:
triplets = '../artistic_sheaf/data/full_triplets.json'
loaded_data = load_json_data(triplets)#[:9914]
print(f"Loaded {len(loaded_data)} triplets from {triplets}")

In [ ]:
# subselect elts in loaded_data where source == generated_i2t
loaded_data_i2t = [elt for elt in loaded_data if elt['source'] == 'generated_i2t']
print(f"Loaded {len(loaded_data_i2t)} triplets from {triplets} with source generated_i2t")
loaded_data_t2i = [elt for elt in loaded_data if elt['source'] == 'generated_t2i']
print(f"Loaded {len(loaded_data_t2i)} triplets from {triplets} with source generated_t2i")

In [ ]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_i2t, preprocess, tokenizer, base_folder='../', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

In [ ]:
clip_images_i2t = []
for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings = model(x_img, x_text, edge_index, edge_attr)
        clip_images_i2t.append(F.normalize(embeddings[: len(edge_attr), :], dim=1))
clip_images_i2t = torch.cat(clip_images_i2t, dim=0)

print(f"Extracted {len(clip_images_i2t)} image embeddings, each of shape {clip_images_i2t[0].shape}")

clip_images_i2t = clip_images_i2t.cpu().detach().numpy()

In [ ]:
# Load test data
test_graph_data, test_node_to_id, test_edge_labels = build_graph_from_json(loaded_data_t2i, preprocess, tokenizer, base_folder='../', split='test')
test_graph_data = test_graph_data.to(device)
print(test_graph_data.edge_index.shape[1])
test_dataset = GraphEdgeDataset(test_graph_data, device=device)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset) // 3, shuffle=False)

In [ ]:
clip_texts_t2i = []

for batch in test_loader:
    with torch.no_grad():
        x_img, x_text, edge_index, edge_attr = process_batch(batch, 'sheaf')
        x_img = x_img.to(device)
        x_text = x_text.to(device)
        edge_index = edge_index.to(device)
        edge_attr = edge_attr.to(device)
        print(x_img.shape, x_text.shape, edge_index.shape, edge_attr.shape)
        
        embeddings = model(x_img, x_text, edge_index, edge_attr)
        clip_texts_t2i.append(F.normalize(embeddings[len(edge_attr):, :], dim=1))
clip_texts_t2i = torch.cat(clip_texts_t2i, dim=0)

print(f"Extracted {len(clip_texts_t2i)} text embeddings, each of shape {clip_texts_t2i[0].shape}")

clip_texts_t2i = clip_texts_t2i.cpu().detach().numpy()

In [138]:
from collections import defaultdict 
def reorder_predictions_by_link_item(
    predictions: np.ndarray,
    new_list,   # "data/full_triplets.json" (the list used to produce predictions)
    old_list,   # the new file with same links but different item2 assignments
    item = 'item2',  # which item to use for matching (default 'item2' for text predictions)
):
    """
    Reorder the text predictions to match the order of (link, item2) in the new triplet file.
    Assumes:
      - predictions_txt[i] corresponds to old_list[i]['item2'] with old_list[i]['link'].
      - Keys used for matching are (link, item2).
      - Handles duplicate (link, item2) by consuming old indices FIFO.
    """
    
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get(item)
        pos_by_key[(link, item2)].append(idx)

    print(len(pos_by_key), "unique (link,item) pairs in the old file.")
    # Build reorder indices to match new_list order
    
    reorder_indices = []
    missing = []
    for tr in new_list:
        key = (tr.get("link"), tr.get(item))
        if pos_by_key[key]:
            reorder_indices.append(pos_by_key[key].pop(0))  # consume one occurrence
        else:
            missing.append(key)

    if missing:
        # Raise for visibility; switch to a warning if partial overlap is expected.
        example = missing[:5]
        print(
            f"{len(missing)} (link,{item}) pairs in the new file were not found in the old predictions. "
            f"Examples: {example}"
        )

    # Reorder predictions
    idx_t = np.array(reorder_indices)
    print(reorder_indices[:10])
    predictions_reordered = predictions[idx_t]
    
    # print how many triplets (link, item1, item2) are the same in the old and new list by creating dictionaries
    # Build mapping: (link, item2) -> queue of old indices
    pos_by_key_all = defaultdict(list)
    for idx, tr in enumerate(old_list):
        link = tr.get("link")
        item2 = tr.get('item2')
        item1 = tr.get('item1')
        pos_by_key_all[(link, item2, item1)].append(idx)

    wrong = []
    for tr in new_list:
        key = (tr.get("link"), tr.get('item2'), tr.get('item1'))
        if pos_by_key_all[key]:
            reorder_indices.append(pos_by_key_all[key].pop(0))  # consume one occurrence
        else:
            wrong.append(key)

    print("Accuracy of preliminary matching (link, item1, item2):",
          1 - len(wrong) / len(new_list))

    return predictions_reordered


In [139]:
loaded_data = load_json_data("data/triplets_semart_test_csv.json")#[:9914]
print(f"Loaded {len(loaded_data)} triplets from data/triplets_semart_test_orig.json")

Loaded 9914 triplets from data/triplets_semart_test_orig.json


In [140]:
predictions_txt_new_order = reorder_predictions_by_link_item(
    clip_texts_t2i,
    new_list=loaded_data_t2i,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item2'
)
print(predictions_txt_new_order.shape)
predictions_txt_new_order[:5]
    

5337 unique (link,item) pairs in the old file.
[0, 13, 27, 42, 74, 103, 112, 121, 130, 139]
Accuracy of preliminary matching (link, item1, item2): 0.08382085939076056
(9914, 512)


array([[-0.00676709,  0.00820725, -0.03396695, ...,  0.01606232,
         0.0491451 ,  0.01856392],
       [ 0.02463759,  0.02067691,  0.02040513, ..., -0.00913673,
         0.07535452, -0.02315984],
       [-0.04231564, -0.04348896,  0.04859691, ..., -0.02238578,
        -0.05060646,  0.00825453],
       [ 0.00445496,  0.025774  ,  0.00216472, ...,  0.01098963,
         0.05871863, -0.03612882],
       [ 0.0236448 ,  0.0134121 , -0.01985466, ...,  0.00667346,
         0.06877115, -0.03030667]], dtype=float32)

In [141]:
predictions_image_new_order = reorder_predictions_by_link_item(
    clip_images_i2t,
    new_list=loaded_data_i2t,  # use only the test portion of the loaded data
    old_list=loaded_data,
    item='item1',
)
print(predictions_image_new_order.shape)

7865 unique (link,item) pairs in the old file.
7397 (link,item1) pairs in the new file were not found in the old predictions. Examples: [('description', 'Images/39651-6eccehom.jpg'), ('description', 'Images/02938-st_luke.jpg'), ('description', 'Images/35872-pastoral.jpg'), ('description', 'Images/25363-bacchus.jpg'), ('description', 'Images/02938-st_luke.jpg')]
[2555, 3981, 7508, 42, 74, 6947, 112, 3264, 130, 6401]
Accuracy of preliminary matching (link, item1, item2): 0.1694573330643534
(2517, 512)


In [147]:
adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data) 
print(f"Adjacency matrix shape: {adj_matrix.shape}")
sim_matrix = get_sim_matrix([t["item1"] + t["link"] for t in loaded_data], 
                            [t["item2"] + t["link"] for t in loaded_data], 
                            clip_images_i2t, clip_texts_t2i,
                            img_to_idx, txt_to_idx)
print(f"Similarity matrix shape: {sim_matrix.shape}")
compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10])

Adjacency matrix shape: (7865, 5337)
Similarity matrix shape: (7865, 5337)


{'t2i_precision@1': tensor(0.0034),
 't2i_recall@1': tensor(0.0030),
 't2i_ndcg@1': tensor(0.0034),
 't2i_precision@5': tensor(0.0041),
 't2i_recall@5': tensor(0.0192),
 't2i_ndcg@5': tensor(0.0117),
 't2i_precision@10': tensor(0.0037),
 't2i_recall@10': tensor(0.0348),
 't2i_ndcg@10': tensor(0.0163),
 'i2t_precision@1': tensor(0.0047),
 'i2t_recall@1': tensor(0.0035),
 'i2t_ndcg@1': tensor(0.0047),
 'i2t_precision@5': tensor(0.0037),
 'i2t_recall@5': tensor(0.0144),
 'i2t_ndcg@5': tensor(0.0094),
 'i2t_precision@10': tensor(0.0039),
 'i2t_recall@10': tensor(0.0302),
 'i2t_ndcg@10': tensor(0.0148),
 'mean_precision@1': tensor(0.0040),
 'mean_recall@1': tensor(0.0033),
 'mean_ndcg@1': tensor(0.0040),
 'mean_precision@5': tensor(0.0039),
 'mean_recall@5': tensor(0.0168),
 'mean_ndcg@5': tensor(0.0106),
 'mean_precision@10': tensor(0.0038),
 'mean_recall@10': tensor(0.0325),
 'mean_ndcg@10': tensor(0.0156)}

In [146]:
verbose = True
for typ in list(set([l['link'] for l in loaded_data])):
    print(f"Processing type: {typ}")
    loaded_data_new = [l for l in loaded_data if l['link'] == typ]
    print(f"Loaded {len(loaded_data_new)} triplets for type {typ}")
    adj_matrix, img_to_idx, txt_to_idx = make_adj_matrix(loaded_data_new)
    print(f"Adjacency matrix shape: {adj_matrix.shape}")
    sim_matrix = get_sim_matrix([t["item1"] + t['link'] for t in loaded_data_new], 
                            [t["item2"] + t['link'] for t in loaded_data_new], 
                            clip_images_i2t, clip_texts_t2i,
                            img_to_idx, txt_to_idx)
    print(f"Similarity matrix shape: {sim_matrix.shape}")
    print(compute_bidirectional_metrics(torch.tensor(sim_matrix), torch.tensor(adj_matrix), k_values=[1, 5, 10]))

    if verbose:
        
        recs = get_top_k_recommendations(torch.Tensor(sim_matrix), k=min(5, len(loaded_data_new)))

        query_field = 'item1'  # image path
        rec_field = 'item2'      # e.g., 'timeframe', 'author', etc.
        idx_to_txt = {idx: txt for txt, idx in txt_to_idx.items()}
        idx_to_img = {idx: img for img, idx in img_to_idx.items()}

        
        for i, rec_indices in enumerate(recs[:5]):  # Show only first 5 for brevity
            query = idx_to_img[i]
            recommendations = [idx_to_txt[j] for j in rec_indices]
            print(f"Query: {'/Users/ludovicaschaerf/Desktop/Sheaf_Art/SemArt/' + query}")
            print(f"Ground Truth: {[loaded_it[rec_field] for loaded_it in loaded_data_new if loaded_it['item1'] + loaded_it['link'] == query]}")
            print("Recommendations:")
            for rec in recommendations:
                print(f"  - {rec}")
            print("-" * 40)

Processing type: description
Loaded 963 triplets for type description
Adjacency matrix shape: (963, 960)
Similarity matrix shape: (963, 960)
{'t2i_precision@1': tensor(0.0135), 't2i_recall@1': tensor(0.0135), 't2i_ndcg@1': tensor(0.0135), 't2i_precision@5': tensor(0.0094), 't2i_recall@5': tensor(0.0469), 't2i_ndcg@5': tensor(0.0296), 't2i_precision@10': tensor(0.0091), 't2i_recall@10': tensor(0.0906), 't2i_ndcg@10': tensor(0.0432), 'i2t_precision@1': tensor(0.0083), 'i2t_recall@1': tensor(0.0083), 'i2t_ndcg@1': tensor(0.0083), 'i2t_precision@5': tensor(0.0091), 'i2t_recall@5': tensor(0.0457), 'i2t_ndcg@5': tensor(0.0260), 'i2t_precision@10': tensor(0.0098), 'i2t_recall@10': tensor(0.0976), 'i2t_ndcg@10': tensor(0.0426), 'mean_precision@1': tensor(0.0109), 'mean_recall@1': tensor(0.0109), 'mean_ndcg@1': tensor(0.0109), 'mean_precision@5': tensor(0.0093), 'mean_recall@5': tensor(0.0463), 'mean_ndcg@5': tensor(0.0278), 'mean_precision@10': tensor(0.0094), 'mean_recall@10': tensor(0.0941),